In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')
print(" All libraries imported successfully!")

 All libraries imported successfully!


In [6]:
from google.colab import files
import io

print("Click 'Choose Files' and select q1_heart_disease.csv from your computer")
uploaded = files.upload()

df = pd.read_csv(io.BytesIO(uploaded['q1_heart_disease.csv']))

print("Shape of dataset:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Value Counts:")
print(df.isnull().sum())
print("\nFirst 5 Rows:")
df.head()

Click 'Choose Files' and select q1_heart_disease.csv from your computer


Saving q1_heart_disease.csv to q1_heart_disease.csv
Shape of dataset: (800, 12)

Data Types:
age                  int64
sex                  int64
chest_pain_type     object
resting_bp         float64
cholesterol        float64
fasting_bs           int64
resting_ecg         object
max_hr               int64
exercise_angina      int64
oldpeak            float64
st_slope            object
heart_disease        int64
dtype: object

Missing Value Counts:
age                 0
sex                 0
chest_pain_type     0
resting_bp         24
cholesterol        32
fasting_bs          0
resting_ecg         0
max_hr              0
exercise_angina     0
oldpeak             0
st_slope            0
heart_disease       0
dtype: int64

First 5 Rows:


,age,sex,chest_pain_type,resting_bp,cholesterol,fasting_bs,resting_ecg,max_hr,exercise_angina,oldpeak,st_slope,heart_disease
0,68,0,atypical_angina,142.0,399.0,0,left_ventricular_hypertrophy,169,0,0.4,up,1
1,58,1,non_anginal,163.0,310.0,1,st_t_wave_abnormality,121,1,1.1,up,1
2,44,1,non_anginal,128.0,175.0,0,normal,183,1,0.2,up,0
3,72,1,asymptomatic,114.0,177.0,0,st_t_wave_abnormality,150,0,1.0,up,1
4,37,1,non_anginal,149.0,271.0,0,normal,136,0,0.4,flat,0


In [10]:
from google.colab import files
import io

print("Click 'Choose Files' and select q1_heart_disease.csv")
uploaded = files.upload()

# This automatically uses whatever filename was uploaded — no hardcoding
filename = list(uploaded.keys())[0]
print("File uploaded as:", filename)

df = pd.read_csv(io.BytesIO(uploaded[filename]))

print("Shape of dataset:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nMissing Value Counts:")
print(df.isnull().sum())
print("\nFirst 5 Rows:")
df.head()

Click 'Choose Files' and select q1_heart_disease.csv


Saving q1_heart_disease.csv to q1_heart_disease (4).csv
File uploaded as: q1_heart_disease (4).csv
Shape of dataset: (800, 12)

Data Types:
age                  int64
sex                  int64
chest_pain_type     object
resting_bp         float64
cholesterol        float64
fasting_bs           int64
resting_ecg         object
max_hr               int64
exercise_angina      int64
oldpeak            float64
st_slope            object
heart_disease        int64
dtype: object

Missing Value Counts:
age                 0
sex                 0
chest_pain_type     0
resting_bp         24
cholesterol        32
fasting_bs          0
resting_ecg         0
max_hr              0
exercise_angina     0
oldpeak             0
st_slope            0
heart_disease       0
dtype: int64

First 5 Rows:


,age,sex,chest_pain_type,resting_bp,cholesterol,fasting_bs,resting_ecg,max_hr,exercise_angina,oldpeak,st_slope,heart_disease
0,68,0,atypical_angina,142.0,399.0,0,left_ventricular_hypertrophy,169,0,0.4,up,1
1,58,1,non_anginal,163.0,310.0,1,st_t_wave_abnormality,121,1,1.1,up,1
2,44,1,non_anginal,128.0,175.0,0,normal,183,1,0.2,up,0
3,72,1,asymptomatic,114.0,177.0,0,st_t_wave_abnormality,150,0,1.0,up,1
4,37,1,non_anginal,149.0,271.0,0,normal,136,0,0.4,flat,0


## EDA Interpretation

**Plot 1 — Target Class Distribution:**
The dataset is fairly balanced between patients with (1) and without (0) heart disease.
A balanced dataset means our models won't be biased toward predicting just one class,
which is important for fair evaluation.

**Plot 2 — Age Distribution:**
Patients with heart disease tend to be slightly older on average. This suggests age
is a meaningful predictor — older patients carry higher risk. However, there is
overlap, so age alone is not sufficient to diagnose heart disease.

**Plot 3 — Correlation Heatmap:**
- `oldpeak` (ST depression) shows a moderate positive correlation with `heart_disease`.
- `max_hr` (maximum heart rate) shows a negative correlation — patients with lower
  max heart rates are more likely to have disease.
- `exercise_angina` also shows a positive correlation with the target.
These insights guide feature selection and confirm medical intuition.
"""

In [11]:
# Step 1: Handle missing values
print("Missing values before:\n", df.isnull().sum())

# Fill numeric columns with median (robust to outliers)
num_cols = df.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical columns with mode
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("\nMissing values after:\n", df.isnull().sum())
print("\n✅ Missing values handled using median (numeric) and mode (categorical)")

Missing values before:
 age                 0
sex                 0
chest_pain_type     0
resting_bp         24
cholesterol        32
fasting_bs          0
resting_ecg         0
max_hr              0
exercise_angina     0
oldpeak             0
st_slope            0
heart_disease       0
dtype: int64

Missing values after:
 age                0
sex                0
chest_pain_type    0
resting_bp         0
cholesterol        0
fasting_bs         0
resting_ecg        0
max_hr             0
exercise_angina    0
oldpeak            0
st_slope           0
heart_disease      0
dtype: int64

✅ Missing values handled using median (numeric) and mode (categorical)


In [12]:
# Step 2: One-hot encode categoricals
df_encoded = pd.get_dummies(df, drop_first=True)
print("Shape after encoding:", df_encoded.shape)
print("Columns:", df_encoded.columns.tolist())

# Step 3: Separate features and target
X = df_encoded.drop('heart_disease', axis=1)
y = df_encoded['heart_disease']

# Step 4: Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Step 5: Train-test split (stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42)

print(f"\n✅ Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Target balance in train: {y_train.value_counts().to_dict()}")


Shape after encoding: (800, 16)
Columns: ['age', 'sex', 'resting_bp', 'cholesterol', 'fasting_bs', 'max_hr', 'exercise_angina', 'oldpeak', 'heart_disease', 'chest_pain_type_atypical_angina', 'chest_pain_type_non_anginal', 'chest_pain_type_typical_angina', 'resting_ecg_normal', 'resting_ecg_st_t_wave_abnormality', 'st_slope_flat', 'st_slope_up']

✅ Train size: 640 | Test size: 160
Target balance in train: {1: 326, 0: 314}
